# Steam Sales Dataset Analysis

This project analyzes the FronkonGames Steam Games Dataset, which contains approximately 124,000 game records with a total size of around 1.39 GB. The dataset includes detailed commercial and performance-related attributes such as pricing, estimated owners, user engagement metrics, reviews, release dates, genres, developers, publishers, and platform availability.

The objective of this analysis is to derive actionable business insights from large-scale marketplace data in order to better understand performance drivers within the Steam ecosystem. The focus areas of this study include:

- Identifying key factors that influence game demand and estimated sales performance  
- Evaluating the relationship between pricing strategies and user engagement  
- Assessing genre, category, and content trends from a market demand perspective  
- Measuring the impact of developers and publishers on commercial success  
- Comparing performance patterns across free-to-play and paid products  

The outcome of this analysis is intended to support data-driven decision-making for stakeholders in game development, publishing, and digital distribution by highlighting trends, opportunities, and performance indicators within the Steam marketplace.

## Import Libraries

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
from pathlib import Path
import importlib
from sales_util import item_parser, item_preprocess
from sales_util.items_data import Item

In [ ]:
# Load env file
load_dotenv(override=True)

In [ ]:
# add dataset location
BASE_DIR = Path(os.getenv("PROJECT_ROOT"))
DATA_PATH = BASE_DIR / os.getenv("DATA_PATH")

In [ ]:
# Logging to huggingface
hf_token = os.environ['HUGGING_KEY']
login(hf_token, add_to_git_credential=True)

## Download the Dataset

In [ ]:
if os.path.exists(DATA_PATH):
    print("Loading dataset from disk...")
    dataset = load_from_disk(DATA_PATH)
else:
    print("Downloading dataset...")
    dataset = load_dataset(
        "FronkonGames/steam-games-dataset",
        split="train",
        trust_remote_code=True
    )
    
    print("Saving dataset to disk...")
    dataset.save_to_disk(DATA_PATH)

print("Dataset ready!")

In [ ]:
print(f"Number of Appliances: {len(dataset):,}")

## Feature Analysis

In [ ]:
data_features = dict(list(dataset.features.items())[:29])
data_features

## Convert To Item Object

In [ ]:
items = []

for i, datapoint in enumerate(tqdm(dataset)):
    items.append(item_parser.parse(datapoint))
    
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

## Summary Statistics

In [ ]:
def describe(arr, name):
    print(f"\n{name}")
    print(f"min: {np.min(arr)}")
    print(f"max: {np.max(arr)}")
    print(f"mean: {np.mean(arr):.2f}")
    print(f"std: {np.std(arr):.2f}")
    print(f"median (p50): {np.percentile(arr, 50):.2f}")
    print(f"p25: {np.percentile(arr, 25):.2f}")
    print(f"p75: {np.percentile(arr, 75):.2f}")
    print(f"p90: {np.percentile(arr, 90):.2f}")
    print(f"p95: {np.percentile(arr, 95):.2f}")
    print(f"p99: {np.percentile(arr, 99):.2f}")

In [ ]:
prices = np.array([item.price for item in items if item.price is not None])
peakCCU = np.array([item.peakCCU for item in items if item.peakCCU is not None])
dlcCount = np.array([item.dlcCount for item in items if item.dlcCount is not None])
positive = np.array([item.positive for item in items if item.positive is not None])
negative = np.array([item.negative for item in items if item.negative is not None])

### Price Distribution

In [ ]:
describe(prices, "== Price ==")

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Price Distribution: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 25, 2))
plt.show()

- Most games are very cheap (median = 2.39)
- Anything above ~30–50 is rare
- There is extrime outlier which is $999, it is clearly garbage or edge-case

### PeakCCU Distribution

In [ ]:
describe(peakCCU, "== peakCCU ==")

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"peakCCU Distribution: Avg {sum(peakCCU)/len(peakCCU):,.2f} and highest {max(peakCCU):,}\n")
plt.xlabel('peakCCU')
plt.ylabel('Count')
plt.hist(peakCCU, rwidth=0.7, color="orange", bins=range(0, 25, 2))
plt.show()

- Most games have no players
- A few huge hits distort everything

### DLC Count Distribution

In [ ]:
describe(dlcCount, "== dlcCount ==")

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"dlcCount Distribution: Avg {sum(dlcCount)/len(dlcCount):,.2f} and highest {max(dlcCount):,}\n")
plt.xlabel('dlcCount')
plt.ylabel('Count')
plt.hist(dlcCount, rwidth=0.7, color="orange", bins=range(0, 25, 2))
plt.show()

- Realistic dlc count range: 0-10

### Positive Comments Count Distribution

In [ ]:
describe(positive, "== positive ==")

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Positive Comments Distribution: Avg {sum(positive)/len(positive):,.2f} and highest {max(positive):,}\n")
plt.xlabel('positive')
plt.ylabel('Count')
plt.hist(positive, rwidth=0.7, color="orange", bins=range(0, 100, 2))
plt.show()

- Most games have very few positive reviews (median = 5)
- A small number of games have extremely large popularity

### Negative Comments Count Distribution

In [ ]:
describe(negative, "== negative ==")

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Negative Comments Distribution: Avg {sum(negative)/len(negative):,.2f} and highest {max(negative):,}\n")
plt.xlabel('negative')
plt.ylabel('Count')
plt.hist(negative, rwidth=0.7, color="orange", bins=range(0, 100, 2))
plt.show()

- Most games have very few negative reviews (median = 1)
- A small number of games receive many negative reviews

## Random Samples

In [ ]:
SIZE = 20_000

In [ ]:
valid_items = [item for item in items if item.price is not None]
prices = np.array([item.price for item in valid_items])

In [ ]:
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w = w / w.sum()

In [ ]:
idx = np.random.choice(len(valid_items), size=SIZE, replace=False, p=w)
samples = [valid_items[i] for i in idx]

In [ ]:
prices_samples = np.array([item.price for item in samples])

In [ ]:
describe(prices_samples, "== prices_samples ==")

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Price Distribution (Weighted): Avg {sum(prices_samples)/len(prices_samples):,.2f} and highest {max(prices_samples):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices_samples, rwidth=0.7, color="orange", bins=range(0, 25, 2))
plt.show()

## Data Preprocess

In [ ]:
importlib.reload(item_preprocess)
data_items = items.copy()

In [ ]:
preprocess_data = []
seen_names = set()  

for item in tqdm(data_items):
    cleaned = item_preprocess.clean_item(item, seen_names)
    
    if cleaned is not None:
        preprocess_data.append(cleaned)
    
print(f"There are {len(preprocess_data):,} preprocess items from {len(items):,} Items")

## Analysis Original and Preprocessed Distribution

In [ ]:
original_prices = np.array([item.price for item in items if item.price is not None])
preprocess_prices = np.array([item.price for item in preprocess_data if item.price is not None])

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

orig_100 = original_prices[:100]
prep_100 = preprocess_prices[:100]

# Original
axs[0].scatter(range(len(orig_100)), orig_100, alpha=0.6, color='blue')
axs[0].set_title("Original Prices")
axs[0].set_xlabel("Index")
axs[0].set_ylabel("Price")

# Preprocessed
axs[1].scatter(range(len(prep_100)), prep_100, alpha=0.6, color='green')
axs[1].set_title("Preprocessed Prices")
axs[1].set_xlabel("Index")
axs[1].set_ylabel("Price")

plt.tight_layout()
plt.show()

## Push To HuggingFace

In [ ]:
random.seed(42)

shuffle_items = preprocess_data.copy()
random.shuffle(shuffle_items)

In [ ]:
username = "KumudithaSilva"
full = f"{username}/items_raw_full"
lite = f"{username}/items_raw_lite"

In [ ]:
train = shuffle_items[:58_729]
val = shuffle_items[58_729:58_729 + 7_341]
test = shuffle_items[58_729 + 7_341:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

Item.push_to_hub(lite, train_lite, val_lite, test_lite)